In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Gaussian Naive Bayes

In [7]:
class GaussianNaiveBayes:
    
    def __init__(self, var_smoothing = 1e-9):
        self.var_smoothing = var_smoothing
        self.classes_ = None
        self.mean_ = {}
        self.var_ = {}
        self.priors_ = {}

    def fit(self, X: np.ndarray | pd.DataFrame, y: np.ndarray):
        """
        Fits the Gaussian Data X
        """

        X = np.asarray(X)
        y = np.asarray(y)

        self.classes_ = np.unique(y)
        n_features = X.shape[1]

        epsilon = self.var_smoothing * X.var(axis = 0).max()

        for c in self.classes_:
            X_c = X[y == c]
            self.mean_[c] = X_c.mean(axis = 0)
            self.var_[c] = X_c.var(axis=0) + epsilon
            self.priors_[c] = X_c.shape[0]/ X.shape[0]
        
    def __log_gaussian_likelihood(self, x, mean, var):
        logged_coef = -0.5 * np.log(2 * np.pi * var)
        logged_exp = -((x - mean)**2) / (2 * var)
        return np.sum(logged_coef + logged_exp)
    
    def predict_log_proba(self, X):
        X = np.asarray(X)
        log_probs = np.zeros((X.shape[0], len(self.classes_)))

        for sample_idx, x in enumerate(X):
            for class_idx, c in enumerate(self.classes_):
                log_prior = np.log(self.priors_[c])
                log_likelihood = self.__log_gaussian_likelihood(x, self.mean_[c], self.var_[c])
                log_probs[sample_idx, class_idx] = log_prior + log_likelihood

        return log_probs

    def predict(self, X):

        log_probs = self.predict_log_proba(X)
        class_indices = np.argmax(log_probs, axis=1)
        return self.classes_[class_indices]

    def predict_proba(self, X):

        log_probs = self.predict_log_proba(X)
        max_log = np.max(log_probs, axis=1, keepdims=True)
        log_probs_shifted = log_probs - max_log
        probs = np.exp(log_probs_shifted)
        probs /= probs.sum(axis=1, keepdims=True)
        return probs

In [11]:
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# My implementation
my_model = GaussianNaiveBayes()
my_model.fit(X_train, y_train)
my_preds = my_model.predict(X_test)

# sklearn's implementation
sk_model = GaussianNB()
sk_model.fit(X_train, y_train)
sk_preds = sk_model.predict(X_test)

print("Match rate:", np.mean(my_preds == sk_preds))  
print("My accuracy:", np.mean(my_preds == y_test))
print("Sklearn accuracy:", np.mean(sk_preds == y_test))

Match rate: 1.0
My accuracy: 0.9777777777777777
Sklearn accuracy: 0.9777777777777777


# Multinomial Naive Bayes

In [12]:
class MultinomialNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.feature_log_prob_ = {}
        self.class_log_prior_ = {}

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_features = X.shape[1]

        for c in self.classes_:
            X_c = X[y == c]

            # count(x_i, y) for each feature i
            feature_counts = np.zeros(n_features)
            for i in range(n_features):
                feature_counts[i] = X_c[:, i].sum()

            total_count = feature_counts.sum()  # count(y)

            log_probs = np.zeros(n_features)
            for i in range(n_features):
                log_probs[i] = np.log(
                    (feature_counts[i] + self.alpha) / (total_count + self.alpha * n_features)
                )

            self.feature_log_prob_[c] = log_probs
            self.class_log_prior_[c] = np.log(X_c.shape[0] / X.shape[0])

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X)
        n_samples = X.shape[0]
        log_probs = np.zeros((n_samples, len(self.classes_)))

        for sample_idx in range(n_samples):
            x = X[sample_idx]
            for class_idx, c in enumerate(self.classes_):
                log_prob = self.class_log_prior_[c]
                # sum over features: x_i * log P(x_i | y)
                for i in range(len(x)):
                    log_prob += x[i] * self.feature_log_prob_[c][i]
                log_probs[sample_idx, class_idx] = log_prob

        return log_probs

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        class_indices = np.argmax(log_probs, axis=1)
        return self.classes_[class_indices]

    def predict_proba(self, X):
        log_probs = self.predict_log_proba(X)
        probs = np.zeros_like(log_probs)
        for sample_idx in range(log_probs.shape[0]):
            row = log_probs[sample_idx]
            max_log = np.max(row)
            shifted = np.exp(row - max_log)
            probs[sample_idx] = shifted / shifted.sum()
        return probs

# Bernoulli Naive Bayes

In [13]:
class BernoulliNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = {}
        self.feature_prob_ = {}

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_features = X.shape[1]

        self.feature_prob_ = {}
        self.class_log_prior_ = {}

        for c in self.classes_:
            X_c = X[y == c]
            n_c = X_c.shape[0]

            probs = np.zeros(n_features)
            for i in range(n_features):
                present_count = X_c[:, i].sum()  # count(x_i=1, y)
                probs[i] = (present_count + self.alpha) / (n_c + 2 * self.alpha)

            self.feature_prob_[c] = probs
            self.class_log_prior_[c] = np.log(n_c / X.shape[0])

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X)
        n_samples = X.shape[0]
        log_probs = np.zeros((n_samples, len(self.classes_)))

        for sample_idx in range(n_samples):
            x = X[sample_idx]
            for class_idx, c in enumerate(self.classes_):
                log_prob = self.class_log_prior_[c]
                p = self.feature_prob_[c]
                # loop over each feature, apply Bernoulli likelihood
                for i in range(len(x)):
                    if x[i] == 1:
                        log_prob += np.log(p[i])
                    else:
                        log_prob += np.log(1 - p[i])
                log_probs[sample_idx, class_idx] = log_prob

        return log_probs

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        class_indices = np.argmax(log_probs, axis=1)
        return self.classes_[class_indices]

    def predict_proba(self, X):
        log_probs = self.predict_log_proba(X)
        probs = np.zeros_like(log_probs)
        for sample_idx in range(log_probs.shape[0]):
            row = log_probs[sample_idx]
            max_log = np.max(row)
            shifted = np.exp(row - max_log)
            probs[sample_idx] = shifted / shifted.sum()
        return probs